In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
import os

# Initialize client using the variables injected from your .env
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)


In [3]:
def llm(prompt):
    # Use the universal chat completions endpoint
    response = openai_client.chat.completions.create(
        model='gpt-4o',  # An actual model string available on GitHub
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    # Return the standard message text
    return response.choices[0].message.content


In [4]:
llm('hi how are you?')

"Hello! I'm just a bunch of code, so I don't have feelings, but I'm here and ready to help you with anything you need. How about you? How's your day going? 😊"

In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [6]:
documents = []
url_prefix ="https://datatalks.club/faq"

for course in courses_raw:
    course_url= f"""{url_prefix}{course["path"]}"""
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()


    documents.extend(course_data)

len(documents)


1342

In [7]:
documents[1000]

{'id': '947bf26d84',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 3. Machine Learning for Classification',
 'question': 'pandas.get_dummies() and DictVectorizer(sparse=False) produce the same type of one-hot encodings:',
 'answer': '<{IMAGE:image_1}>\n\n`DictVectorizer(sparse=True)` produces [CSR](https://en.wikipedia.org/wiki/Sparse_matrix#Compressed_sparse_row_(CSR,_CRS_or_Yale_format)) format, which is both more memory efficient and converges better during fit().\n\nIt stores non-zero values and indices instead of adding a column for each class of each feature, which can result in large numbers of columns (e.g., models of cars).\n\nUsing "sparse" format is slower (around 6-8 minutes for Q6 task - Linear/Ridge Regression) for a high number of classes (like car models) and produces slightly worse results in both Logistic and Linear/Ridge Regression.\n\nIt also generates convergence warnings for Linear/Ridge Regression.'}

In [8]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [9]:
def search(question, course="llm-zoomcamp"):
    boost_dict={"question": 2.0, "section": 0.5}
    filter_dict={"course": course}
    search_results = index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
)

    return search_results

In [10]:
question = "I just discovered the course. Can I join now?"
search_results = search(question)


In [11]:
search_results


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [12]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [13]:
USER_PROMPT_TEMPLATE = """
Question:
{question_}

Context:
{context_}
"""

In [14]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: "+doc["question"])
        lines.append("A:" + doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()

In [15]:
def build_prompt(question, search_results):
    context=build_context(search_results)

    prompt = USER_PROMPT_TEMPLATE.format(
        question_= question,
        context_= context
    )
    return prompt.strip()


In [16]:
prompt = build_prompt(question,search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A:Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A:You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A:No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.



In [17]:
response = openai_client.chat.completions.create(
        model='gpt-4o',  # An actual model string available on GitHub
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    # Return the standard message text
response.choices[0].message.content

'Yes, you can still join the course. If you’re aiming to receive a certificate, you need to ensure that you submit your Capstone project while the submission form is still open. Certificates are only awarded to participants who complete the course with a live cohort, as the peer-review process for the Capstone projects happens during this time. If you’ve missed earlier parts of the course, don’t worry—homework is not mandatory for the certificate, though completing it can be helpful for mastering the concepts.'

In [18]:
response.choices

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you can still join the course. If you’re aiming to receive a certificate, you need to ensure that you submit your Capstone project while the submission form is still open. Certificates are only awarded to participants who complete the course with a live cohort, as the peer-review process for the Capstone projects happens during this time. If you’ve missed earlier parts of the course, don’t worry—homework is not mandatory for the certificate, though completing it can be helpful for mastering the concepts.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, '

In [19]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=message_history
)


In [20]:
response.choices

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still open.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})]

In [21]:
def llm(instructions, user_prompt, model): # Changed _model to model
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt} 
    ]  
    response = openai_client.chat.completions.create(
        model=model,       # Matches the function parameter
        messages=message_history # Fixed typo: 'messages' instead of 'message'
    ) 
    return response.choices[0].message.content

def rag(query, _model="gpt-4o"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    # This now works perfectly because llm expects 'model='
    answer = llm(INSTRUCTIONS, prompt, model=_model) 
    return answer


In [22]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [23]:
import os
from dotenv import load_dotenv
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI


load_dotenv()

# Initialize client using the variables injected from your .env
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

documents = load_faq_data()
index = build_index(documents)



assistant = RAGBase(
    index=index,
    llm_client=openai_client
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [24]:
answer = assistant.rag("do you have a llm-zoomcamp?")
print(answer)

Yes, there is an LLM Zoomcamp. Based on the provided context, it is a course related to large language models, and you do not need a confirmation email to start as you are already accepted upon registration. You can begin learning and submitting homework right away while the registration form is open.
